#### 0. Change directory

In [1]:
# Reference: https://stackoverflow.com/questions/15514593/importerror-no-module-named-when-trying-to-run-python-script/15622021#15622021
# Here choosing the folder that stores all the modelling modules
import sys
sys.path.append(r'S:\\ML_simulation_uncertainty_esk\\Modelling')

#### 1. Packages

In [2]:
from folder import *
from randomisation import random_values_generation
from dataPreparation import download_lidar, \
                            padding_combination, \
                            dem_raster_reference, \
                            terrain_shading             # For lidar downloading, padding calculation, dem reference generation, and terrain shading

from transformation import center_calculation, get_url, get_tile_files, \
                           MyPool, transformation_performance, transformation_parallelism # For transformation using multiprocessing

from rasterSimulation import raster_generation

from floodModel import tide_flow_data, bdy_generation, tideboundary_points, flood_simulation

# For creating plot
import matplotlib.pyplot

# For untransformation
from untransformation import bathy_untransformation_parallelism

import multiprocessing
from functools import partial


# For terrain shading
import os
import json
import rioxarray as rxr
import xarray as xr
from osgeo import gdal
import numpy as np

#### 2. Necessary variables

In [3]:
# 20-m resolution
# # Construct a list of boundary
# resolution = 20
# number_pixel_x = 16 * 21
# number_pixel_y = 16 * 14

# xmin = 1767790
# ymin = 5471400

In [4]:
# 10-m resolution
# Construct a list of boundary
resolution = 10*2
number_pixel_x = 16 * (50/2)
number_pixel_y = 16 * (40/2)

xmin = 1929005
ymin = 5629885

In [5]:
# 5-m resolution
# # Construct a list of boundary
# resolution = 5
# number_pixel_x = 16 * 84
# number_pixel_y = 16 * 56

# xmin = 1767790
# ymin = 5471400

In [6]:
# 2-m resolution
# # Construct a list of boundary
# resolution = 2
# number_pixel_x = 16 * 210
# number_pixel_y = 16 * 140

# xmin = 1767790
# ymin = 5471400

##### 2.2. Other basic variables

In [7]:
xmax = xmin + resolution * number_pixel_x
ymax = ymin + resolution * number_pixel_y

# Boundary for the area of interest
boundary_1 = [xmin, ymin, xmax, ymax]

# Padding boundary for DEMs (use this boundary to create padding)
addition_2 = 16 * 3
boundary_2 = padding_combination(boundary_1, addition_2)

# Boundary for tiles (use this boundary to download LiDAR)
addition_3 = 16 * 3
boundary_3 = padding_combination(boundary_2, addition_3)

In [8]:
# LiDAR dataset name
lidar_name = "NZ20_Hawkes"

# # Chunks and processors information (for computer has 8 cores)
# # For 10-m resolution
# size_of_chunk = 100
# size_of_processor = 4
# For 20-m resolution
size_of_chunk = 50
size_of_processor = 2
# # For 2-m resolution
# size_of_chunk = 400
# size_of_processor = 6
# # For 5-m resolution
# size_of_chunk = 200
# size_of_processor = 6

# Flow data
flow_path = fr"S:\Combine_Uncertainty_Folders\uncertainty_combination\esk\vers001\other_data\flow.csv"

# Tide data
tide_path = fr"S:\Combine_Uncertainty_Folders\uncertainty_combination\esk\vers001\other_data\tide.csv"

# date start and end
date_start = r"2023-02-11 22:00:00"
date_end = r"2023-02-15 15:00:00"

# Tide flow data
tide_flow_data(tide_path, flow_path, date_start, date_end)

# Extracting flowdepth rate (for flowdepth_extraction() function)
flowdepth_rate = 0

# Time to extract water (for water_extraction() function)
extract_wd = "out.max"
extract_wse = "out.mxe"
extract_elev = "elev"

##### 2.3. Preparing flood model parameters

In [9]:
# General parameters
general_para = {
    ## GEOFABRICS
    # Land
    "land": 51559,
    # Bathymetry
    "ocean_contours": 50448,
    
    # Waterways --------
    "drain": 5,
    "stream": 7.5,
    "river": 10,
    
    # Osm
    "osm_id": 109375060,
    "osm_date": "2023-02-02T00:00:00Z",
    
    # River features
    "max_channel_width": 200,
    "min_channel_width": 15,
    "max_bank_height": 4,
    "min_bank_height": 0.5,
    
    # Others
    "network_id": 8021954,
    "area_threshold": 249500000,
    "width_centre_smoothing": 10,
    "river_corridor_width": 300,
    "cross_section_spacing": 10,
    "rec_alignment_tolerance": 65,
    "veg_lidar_classifications_to_keep": [2, 3, 4, 5, 9],
    
    # File
    "network_file": "river_network_esk.geojson",
    
    ## TRANSFORMATION
    'offset_x': 1900000,
    'offset_y': 5600000,
    
    
    ## FLOOD MODEL
    "tide_sampledis": 10, # 10 or 10
    "tide_num_points": 13, # 652 or 13
    "sim_time": 320400,
    
    # SEA FILTER
    'sea_catchment': 0
}

# Write out json
with open(fr"{other_data}/general_parameters.json", "w") as general_parameters_file:
    json.dump(general_para, general_parameters_file)

In [10]:
# Create tide_flow data
tide_flow_data(tide_path, flow_path, date_start, date_end)

# River points
river_points = (1929029.16, 5633440.46)

# Create BDY file
domain_name = "Esk"
bdy_generation(resolution, domain_name)

# Create tide boundary
tideboundary_points(boundary_1, False)

##### 2.4. Preparing some flood model inputs

In [11]:
%%time
# # Download LiDAR
# download_lidar(boundary_3, lidar_name)

CPU times: total: 0 ns
Wall time: 0 ns


In [12]:
# %%time
# # Reference DEM without padding
# dem_raster_reference(resolution, size_of_chunk, size_of_processor,
#                      boundary_1, lidar_name, "no_padding")

In [13]:
# %%time
# # Reference DEM with padding
# dem_raster_reference(resolution, size_of_chunk, size_of_processor,
#                      boundary_2, lidar_name, "padding")

In [14]:
# %%time
# # DEM for hillshade/terrain shade
# dem_raster_reference(5, 200, 4,
#                      boundary_1, lidar_name, "shading")

# # Terrain shading
# terrain_shading(50, 355)

In [15]:
# Simulation quantity
sim_quantity = 100

# Calculate coordinates of center point
center_point = center_calculation(True)
center_x = center_point[0]                     # Extract x coordinate of center point
center_y = center_point[1]                     # Extract y coordinate of center point

# Get url list file
url_list_file = get_url(lidar_name)
tile_files = get_tile_files(lidar_name)

#### 3. Bathymetry and transformation preparations

##### 3.1. Bathymetry kind and para name

In [16]:
# from bedSim import generate_bedsim, generate_bedsim_combination

In [17]:
kind = 'rupp'
para_name = 'com'
ran_bathy = [num for num in range(1, sim_quantity+1)]

##### 3.2. Random transformations

##### 3.2.1. E translation (x)

In [18]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     50,
#     [0, 0],
#     [0, resolution/2],
#     [0, 0],
#     'uniform',
#     True
# )

##### 3.2.2. N translation (y)

In [19]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     50,
#     [0, 0],
#     [0, 0],
#     [0, resolution/2],
#     'uniform',
#     True
# )

##### 3.2.3. N-E translation

In [20]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     50,
#     [0, 0],
#     [0, resolution/2],
#     [0, resolution/2],
#     'uniform',
#     True
# )

##### 3.2.4. Rotation

In [21]:
# # Transformation values
# ran_trans = random_values_generation(
#     1,
#     50,
#     [0, 90],
#     [0, 0],
#     [0, 0],
#     'uniform',
#     True
# )

##### 3.2.5. N-E translation and rotation

In [22]:
# Transformation values
ran_trans = random_values_generation(
    1,
    sim_quantity,
    [0, 90],
    [0, resolution/2],
    [0, resolution/2],
    'uniform',
    True
)

##### 3.2.6. Only for 2-m resolution (if not skip this part)

In [23]:
# import numpy as np
# ran_trans = np.delete(ran_trans, [3, 4, 12, 40, 41], 0)
# ran_trans = ran_trans[:50]

#### 3.2.7. Combine random values

In [24]:
ran_bathy_trans = np.c_[ran_bathy, ran_trans]

In [25]:
# ran_bathy_trans = np.delete(
#     ran_bathy_trans, 
#     [7, 8, 9, 10, 28, 29, 30, 31, 32, 35, 36, 37, 38,
#      39, 40, 41, 49, 56, 77, 78, 79, 80, 84, 98, 99], 
#     axis=0
# )

In [26]:
ran_bathy_trans = ran_bathy_trans[[
    3, 5, 6, 11, 16, 17, 18, 19,
    34, 42, 43, 54, 55, 57, 63, 64, 65, 69,
    74, 75, 76, 81, 82, 83, 85, 86, 91, 92, 93, 94, 96, 97
]]

#### 4. Execution

In [ ]:
# %%time
# generate_bedsim(para_name, kind, sim_quantity)
# generate_bedsim_combination(kind, sim_quantity)

In [ ]:
# # Ref: https://gis.stackexchange.com/questions/326968/ogr2ogr-error-1-proj-pj-obj-create-cannot-find-proj-db
# import os
# os.environ['PROJ_LIB'] = "C:\Program Files\Anaconda3\envs\geofabrics49\Library\share\proj"

In [ ]:
# %%time
# transformation_parallelism(
#     kind, para_name,
#     center_x,
#     center_y,
#     lidar_name,
#     tile_files,
#     30,
#     url_list_file,
#     ran_bathy_trans,
#     1
# )

In [ ]:
# for ran in ran_bathy_trans:
#     raster_generation(
#         kind, para_name,
#         resolution, 
#         size_of_chunk, size_of_processor,
#         boundary_2, lidar_name,
#         center_x, center_y,
#         ran
#     )

In [28]:
%%time
# DEM CREATION
# List parameters
kind_func = kind
para_name_func = para_name
resolution_func = resolution
chunk_size_func = size_of_chunk
processor_func = size_of_processor
padding_func = boundary_2
lidar_dataset_name = lidar_name

# Design func
func = partial(
    raster_generation,
    kind_func, para_name_func,
    resolution_func,
    chunk_size_func,
    processor_func,
    padding_func,
    lidar_dataset_name,
    center_x, center_y
)

# Design pool
pool = MyPool(4)
pool.map(func, [ran for ran in ran_bathy_trans])
pool.close()
pool.join()

AttributeError: 'NoneType' object has no attribute 'lock_acquire'

In [ ]:
# %%time
# for ran_bathy_trans_i in ran_bathy_trans:
#     flood_simulation(
#         domain_name,
#         river_points,
#         kind, para_name,
#         center_x, center_y,
#         fr"S:\\new_versions\\data",
#         ran_bathy_trans_i
#     )

In [ ]:
%%time
# Water depth
bathy_untransformation_parallelism(
    extract_wd,
    ran_bathy_trans,
    kind, para_name,
    10
)

In [ ]:
%%time
# Water surface elevation
bathy_untransformation_parallelism(
    extract_wse,
    ran_bathy_trans,
    kind, para_name,
    10
)

In [ ]:
%%time
# Elevation
bathy_untransformation_parallelism(
    extract_elev,
    ran_bathy_trans,
    kind, para_name,
    10
)